# Scenario C3: Single Supplier — REC + Battery Optimization (Mixed Prosumers)

**Description:** Single supplier mandate with REC and centralized battery
storage optimisation. A **two-stage hierarchical optimization** is solved as a
Mixed-Integer Linear Programme (MILP) using **Pyomo + Gurobi** to minimise
total community energy costs while guaranteeing Pareto-fair individual outcomes.
Prosumers are modelled as **mixed** (local load + RES generation + storage).

**Two-stage approach:**
- **Stage 1 — Individual baseline:** Each participant independently minimizes
  their own energy bill (prosumers dispatch batteries to maximise self-consumption;
  consumers face fixed grid costs). The resulting individual costs $C^{\mathrm{ind}*}_i$
  serve as cost caps for Stage 2.
- **Stage 2 — Collective REC optimization:** All batteries and internal energy
  trades are co-optimised across the community to minimise aggregate cost, subject
  to a **Pareto fairness constraint** ($C^{\mathrm{rec}}_i \leq C^{\mathrm{ind}*}_i\ \forall i$).

**Internal pricing — feed-in parity:**
$$p^{\mathrm{p2p}}_t = p^{\mathrm{exp}}_t$$

The internal REC transfer price $p^{\mathrm{p2p}}_t$ is anchored to the grid
feed-in (export) tariff $p^{\mathrm{exp}}_t$ at every 15-minute interval.
This pricing rule has distinct implications for each participant type:

- **Prosumer indifference.** A prosumer with surplus PV generation faces two
  options: (i) export to the grid at $p^{\mathrm{exp}}_t$, or (ii) sell to a
  fellow REC member at $p^{\mathrm{p2p}}_t$. Because
  $p^{\mathrm{p2p}}_t = p^{\mathrm{exp}}_t$, the per-kWh revenue is identical
  under both routes — the prosumer is **economically indifferent** between
  internal and external sales. Consequently, REC membership never reduces a
  prosumer's revenue; participation is costless in the worst case and
  beneficial whenever the Pareto constraint yields additional savings.

- **Consumer incentive.** The Austrian retail import tariff
  $p^{\mathrm{imp}}_{i,t}$ exceeds the feed-in price:
  $p^{\mathrm{imp}}_{i,t} > p^{\mathrm{exp}}_t\ \forall\, t$. A consumer
  purchasing energy internally therefore pays $p^{\mathrm{exp}}_t$ instead of
  $p^{\mathrm{imp}}_{i,t}$, saving
  $(p^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_t)\,\Delta t$ € per kWh
  traded. This price wedge provides a **clear economic incentive** for
  consumers to join the REC and source as much load as possible from
  internal generation.

- **Aggregate neutrality.** At the community level, internal transfers are
  zero-sum: every internal sale by a prosumer is matched by an internal
  purchase by a consumer. The $p^{\mathrm{p2p}}_t$ terms cancel in the aggregate
  community cost, so the transfer price affects only the **distribution** of
  savings among members — not the total cost minimised by the MILP. The
  Pareto fairness constraint then ensures no individual is made worse off by
  that distribution.

- **Why feed-in parity?** Alternative pricing rules (e.g., mid-market
  $p^{\mathrm{mid}}_t = \tfrac{1}{2}(p^{\mathrm{imp}}_t + p^{\mathrm{exp}}_t)$)
  would increase prosumer revenue above the grid feed-in level, creating
  a dependency on REC participation that may not persist if membership
  changes. Feed-in parity is the most conservative rule: it maximises
  the consumer share of savings while preserving prosumer indifference,
  requiring no additional negotiation or dynamic price-setting mechanism.

**Battery assets:**
| Asset | Capacity | Max Power | Efficiency | SoC Range |
|---|---|---|---|---|
| BESS_REC_01 — Community battery | 200 kWh | 68 kW | 95% | 20–100% |
| BESS_REC_02 — Prosumer residential | 80 kWh | 25 kW | 95% | 20–100% |

**Configuration:**
| Parameter | Value |
|---|---|
| Suppliers | 1 — SUP_A |
| Balancing Groups | 1 — BG_A (all 9 participants) |
| RECs | 1 — REC_01 (all 9 members, proportional sharing) |
| Prosumer type | **Mixed** (load + RES generation + storage) |
| REC incentives | Shared energy: 0.01 €/kWh · Self-consumption: 0.015 €/kWh |
| Optimization | Two-stage MILP · Pyomo + Gurobi · Pareto fairness guarantee |
| MPC horizon | 24 h rolling (96 × 15-min steps), re-solved daily |
| Market type | day_ahead_with_intraday_updates |
| Battery | 2 assets (community + residential) |

**Research role:** Quantifies the marginal value of battery storage on top of a
REC under a single-supplier mandate. Storage allows temporal shifting of PV surplus
into high-demand periods, increasing internal sharing ratios and reducing balancing
exposure.

**Comparison pairs (mixed-prosumer track):**
- C1 vs A2-mixed → marginal value of battery storage on top of a single-supplier REC
- C1 vs A1-mixed → combined effect of REC + battery vs no-REC baseline
- C1 vs B2-mixed → single-supplier + battery vs multi-supplier REC (no battery)

## 1. Import Dependencies

The simulation is driven by the `EnergyMarketOperations` class, which encapsulates the full Austrian energy-market pipeline — day-ahead scheduling, intraday adjustments, battery MILP dispatch, REC settlement, balancing settlement, and supplier billing — in a single callable object.

In [ ]:
from energy_market_operations import EnergyMarketOperations

## 2. Initialize Pipeline

The pipeline is instantiated from the C3 scenario JSON, which specifies the nine participants (three mixed prosumers with PV + BESS, six consumers), the single-supplier market structure (SUP_A / BG_A), REC membership and proportional sharing rules, two battery assets (BESS_REC_01 community + BESS_REC_02 residential), dynamic wholesale prices, and the SimBench `1-LV-rural3--2-no_sw` network topology.  All time series — load, PV generation, day-ahead and intraday forecasts, and market prices — are loaded at 15-minute resolution for the full 2016 calendar year (35 136 time steps).

In [ ]:
CONFIG_FILE = "C3_single_supplier_rec_battery.json"
pipe = EnergyMarketOperations(CONFIG_FILE, scenario_name="C3")

## 3. Run Full Pipeline

The `run_all()` method executes the complete market simulation in sequence:

1. **Day-Ahead Market** — Each participant's net position is scheduled using day-ahead price and generation/load forecasts.
2. **Intraday Market** — Positions are revised with updated intraday forecasts to reduce expected imbalances.
3. **Battery Optimization** — A two-stage MILP (Pyomo + Gurobi) is solved over a rolling 24-hour MPC horizon.  Stage 1 computes each participant's individual baseline cost $C^{\mathrm{ind}*}_i$; Stage 2 co-optimises all batteries and internal energy trades to minimise aggregate REC cost subject to the Pareto fairness constraint $C^{\mathrm{rec}}_i \leq C^{\mathrm{ind}*}_i$.  The internal REC price is set at feed-in parity ($p^{\mathrm{p2p}}_t = p^{\mathrm{exp}}_t$).
4. **REC Settlement** — Internally shared energy is allocated proportionally among community members; reduced grid fees apply to shared volumes.
5. **Balancing Market** — Residual imbalances between scheduled positions and metered actuals are settled at dual imbalance prices.
6. **Supplier Billing** — Final participant costs are computed, incorporating DA/ID market costs, balancing charges, REC credits, and grid fees.

## Two-Stage MILP Formulation for REC Battery Optimization

The battery dispatch is determined by solving a **two-stage Mixed-Integer Linear Programme (MILP)** embedded in a rolling MPC framework (24-hour horizon, 96 × 15-min steps, re-solved daily).

---

### Sets, indices, and parameters

| Symbol | Description |
|--------|-------------|
| $\mathcal{T} = \{0,\dots,T{-}1\}$ | Time steps in one rolling block ($T = 96$) |
| $\mathcal{P}$ | Set of prosumers (3 members with PV + BESS) |
| $\mathcal{C}$ | Set of consumers (6 members, load only) |
| $\mathcal{B}$ | Set of batteries in the REC |
| $\Delta t = 0.25\,\text{h}$ | Step duration |
| $L_{i,t}$, $G_{i,t}$ | Load (kW) and PV generation (kW) for participant $i$ |
| $p^{\mathrm{imp}}_{i,t}$, $p^{\mathrm{exp}}_{i,t}$ | Grid import tariff / feed-in price (€/kWh) |
| $p^{\mathrm{p2p}}_t = p^{\mathrm{exp}}_t$ | Internal REC price — set at **feed-in parity** |
| $p^{\mathrm{grid}}_t$ | Reduced grid access fee for internally shared energy (€/kWh) |
| $\overline{P}^{\mathrm{ch}}_b$, $\overline{P}^{\mathrm{dis}}_b$ | Max charge / discharge power (kW) for battery $b$ |
| $\eta^{\mathrm{ch}}_b$, $\eta^{\mathrm{dis}}_b$ | Charge / discharge efficiencies |
| $E^{\min}_b$, $E^{\max}_b$ | SOC bounds (kWh) |
| $E^{0}_b$ | Initial SOC (terminal SOC from previous rolling block) |

---

### Stage 1 — Individual Optimization (Baseline Cost Caps)

Each participant $i \in \mathcal{C} \cup \mathcal{P}$ independently minimizes their energy bill:

$$\min_{x^{\mathrm{ch}}, x^{\mathrm{dis}}, P^{\mathrm{imp}}, P^{\mathrm{exp}}, E} \sum_{t \in \mathcal{T}} \Delta t \left( p^{\mathrm{imp}}_{i,t}\, P^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_{i,t}\, P^{\mathrm{exp}}_{i,t} \right)$$

**Prosumer energy balance:**

$$P^{\mathrm{imp}}_{i,t} - P^{\mathrm{exp}}_{i,t} + x^{\mathrm{dis}}_{b,t} - x^{\mathrm{ch}}_{b,t} = L_{i,t} - G_{i,t} \qquad \forall\, t,\; i \in \mathcal{P}$$

**Consumer energy balance:**

$$P^{\mathrm{imp}}_{i,t} = L_{i,t}, \quad P^{\mathrm{exp}}_{i,t} = 0 \qquad \forall\, t,\; i \in \mathcal{C}$$

**SOC dynamics, bounds, and no-simultaneous charge/discharge:**

$$E_{b,t} = E_{b,t-1} + \eta^{\mathrm{ch}}_b\, x^{\mathrm{ch}}_{b,t}\, \Delta t - \frac{x^{\mathrm{dis}}_{b,t}}{\eta^{\mathrm{dis}}_b}\, \Delta t, \qquad E^{\min}_b \le E_{b,t} \le E^{\max}_b$$

$$x^{\mathrm{ch}}_{b,t} \le \overline{P}^{\mathrm{ch}}_b\, u_{b,t}, \qquad x^{\mathrm{dis}}_{b,t} \le \overline{P}^{\mathrm{dis}}_b\, (1 - u_{b,t}), \qquad u_{b,t} \in \{0,1\}$$

**Outcome:** Individual minimum cost $C^{\mathrm{ind}*}_i$ for each participant — serves as the **cost cap** in Stage 2.

---

### Stage 2 — Collective REC Optimization (Community Cost Minimization + Fairness)

The REC collectively minimizes aggregate cost across all participants:

$$\min \sum_{i \in \mathcal{C} \cup \mathcal{P}} \sum_{t \in \mathcal{T}} \Delta t \left[ p^{\mathrm{imp}}_{i,t}\, P^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_{i,t}\, P^{\mathrm{exp}}_{i,t} + p^{\mathrm{grid}}_t\, E^{\mathrm{self}}_{i,t} \right]$$

**Modified prosumer energy balance (with internal trades):**

$$P^{\mathrm{imp}}_{i,t} - P^{\mathrm{exp}}_{i,t} + x^{\mathrm{dis}}_{b,t} - x^{\mathrm{ch}}_{b,t} = L_{i,t} - G_{i,t} - \sum_{j \neq i} E^{\mathrm{sale}}_{i,j,t} + \sum_{j \neq i} E^{\mathrm{purch}}_{i,j,t}$$

**Modified consumer energy balance:**

$$P^{\mathrm{imp}}_{i,t} = L_{i,t} - \sum_{j \in \mathcal{P}} E^{\mathrm{purch}}_{i,j,t}$$

**Internal trade balance:**

$$\sum_{j \neq i} E^{\mathrm{sale}}_{i,j,t} = \sum_{j \neq i} E^{\mathrm{purch}}_{j,i,t} \qquad \forall\, t$$

**Pareto fairness constraint** — no participant pays more than their Stage 1 baseline:

$$C^{\mathrm{rec}}_i = \sum_{t \in \mathcal{T}} \Delta t \left[ p^{\mathrm{imp}}_{i,t}\, P^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_{i,t}\, P^{\mathrm{exp}}_{i,t} + p^{\mathrm{p2p}}_t \left( \sum_j E^{\mathrm{purch}}_{i,j,t} - \sum_j E^{\mathrm{sale}}_{i,j,t} \right) \right] \leq C^{\mathrm{ind}*}_i \quad \forall\, i$$

**Battery constraints** (identical to Stage 1): SOC dynamics, SOC bounds, power limits, no-simultaneous charge/discharge, plus a **terminal SOC constraint** to prevent drift:

$$E_{b,T-1} \geq E^{0}_b \qquad \forall\, b \in \mathcal{B}$$

---

### Internal Pricing — Feed-In Parity

The internal REC price is set equal to the feed-in (export) price: $p^{\mathrm{p2p}}_t = p^{\mathrm{exp}}_t$

- **Prosumer indifference:** Revenue per kWh of surplus is $p^{\mathrm{exp}}_t$ whether sold internally or exported to the grid — selling to the REC or to the supplier is economically equivalent. Prosumers are **never disadvantaged** by REC membership.
- **Consumer incentive:** The internal purchase price $p^{\mathrm{exp}}_t$ is strictly below the retail import tariff $p^{\mathrm{imp}}_t$, so every internally purchased kWh saves the consumer $(p^{\mathrm{imp}}_t - p^{\mathrm{exp}}_t) \cdot \Delta t$ €. This creates a clear economic incentive to join the REC.

---

### Rolling-Horizon MPC Execution

The two-stage MILP is solved over a 24-hour block and repeated daily: the terminal SOC $E_{b,T-1}$ from block $k$ becomes $E^{0}_b$ for block $k{+}1$, propagating battery state across the full simulation year.

In [ ]:
pipe.run_all()

## 4. Financial Summary

The aggregated financial summary reports total revenues (customer electricity sales), total procurement costs (DA + ID market purchases, balancing charges), and the resulting supplier profit or loss.  Because all nine participants are assigned to a single supplier (SUP_A) and a single balancing group (BG_A), the summary reflects the consolidated position of the entire community.  Battery optimisation revenues appear as reduced grid import costs and increased internal sharing credits.

In [ ]:
pipe.summary()

## 5. Financial Visualisation

The stacked bar chart decomposes the supplier's annual financial position into its constituent revenue and cost categories: day-ahead procurement, intraday adjustment costs, balancing charges (positive and negative imbalances), and customer sales revenue.  Comparing these bars against the A2-mixed scenario (REC without battery) isolates the monetary contribution of battery dispatch — primarily visible as a reduction in DA procurement costs and balancing exposure.

In [ ]:
pipe.plot_financials()

## 6. Imbalance Analysis

System imbalances measure the deviation between the supplier's scheduled DA/ID position and the actual metered energy at each 15-minute interval.  In the C3 scenario the battery MILP is optimised against intraday forecasts, so residual imbalances arise primarily from forecast errors in PV generation and load.  The time-series plot reveals seasonal patterns — larger imbalances during summer (high PV volatility) and shoulder months — and quantifies the extent to which battery storage dampens balancing exposure relative to the non-battery A2-mixed baseline.

In [ ]:
pipe.plot_imbalances()

## 7. Battery Optimisation Results

For each of the two battery assets (BESS_REC_01 community-scale, 200 kWh; BESS_REC_02 residential, 80 kWh), a two-panel figure is produced:

- **Upper panel — State of Charge (SOC):**  The SOC trajectory over the full calendar year, bounded by $E^{\min}_b = 0.20\, E^{\mathrm{cap}}$ (orange dashed) and $E^{\max}_b = E^{\mathrm{cap}}$ (red dashed).  The terminal-SOC constraint ($E_{b,T-1} \geq E^{0}_b$) prevents the MILP from systematically depleting the battery across rolling blocks.
- **Lower panel — Charge / Discharge Power:**  Positive bars indicate charging (green, capped at $\overline{P}^{\mathrm{ch}}_b$); negative bars indicate discharging (red, capped at $\overline{P}^{\mathrm{dis}}_b$).  The daily cycling pattern reflects the two-stage MILP strategy: charging during midday PV surplus (low-price windows) and discharging during evening demand peaks (high-price windows).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Normalise: build {battery_id: (schedule_df, tech_params)} from all available batteries
batteries_raw = pipe.config.get('batteries', pipe.config.get('battery_storage'))
if isinstance(batteries_raw, dict):
    batteries_raw = [batteries_raw]

battery_cfgs = {b['battery_id']: b.get('technical_parameters', {}) for b in batteries_raw}

# Use per-battery schedules if available, fall back to single aggregated schedule
schedules = getattr(pipe, 'battery_schedules', None) or {}
if not schedules and hasattr(pipe, 'battery_schedule_df') and not pipe.battery_schedule_df.empty:
    # Single-battery backward compat
    batt_id = list(battery_cfgs.keys())[0]
    schedules = {batt_id: pipe.battery_schedule_df}

if not schedules:
    print("Battery optimization results not available")
else:
    for batt_id, df in schedules.items():
        tech = battery_cfgs.get(batt_id, {})
        capacity_kwh     = tech.get('capacity_kwh', 200)
        max_charge_kw    = tech.get('max_charge_power_kw', 68)
        max_discharge_kw = tech.get('max_discharge_power_kw', 68)
        soc_min_kwh      = capacity_kwh * tech.get('soc_min_percent', 20)  / 100.0
        soc_max_kwh      = capacity_kwh * tech.get('soc_max_percent', 100) / 100.0
        initial_soc_kwh  = capacity_kwh * tech.get('initial_soc_percent', 50) / 100.0

        batt_name = next((b['battery_name'] for b in batteries_raw if b['battery_id'] == batt_id), batt_id)

        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        fig.suptitle(f'{batt_id} — {batt_name}', fontsize=12, fontweight='bold')

        dates = df.index

        # SOC with reference lines
        axes[0].plot(dates, df['soc_kwh'], label='SOC', color='blue', linewidth=0.8)
        axes[0].axhline(soc_max_kwh,     color='red',    linestyle='--', linewidth=1.0,
                        label=f'SOC max ({soc_max_kwh:.0f} kWh)')
        axes[0].axhline(soc_min_kwh,     color='orange', linestyle='--', linewidth=1.0,
                        label=f'SOC min ({soc_min_kwh:.0f} kWh)')
        axes[0].axhline(initial_soc_kwh, color='gray',   linestyle=':',  linewidth=1.0,
                        label=f'Initial SOC ({initial_soc_kwh:.0f} kWh)')
        axes[0].set_ylabel('SOC (kWh)')
        axes[0].legend(loc='upper right', fontsize=8)
        axes[0].set_title(f'State of Charge  [capacity: {capacity_kwh} kWh]')
        axes[0].grid(True, alpha=0.3)

        # Charge/discharge with power limit lines
        axes[1].bar(dates, df['charge_kw'],
                    width=pd.Timedelta('15min'), color='green', alpha=0.7, label='Charge')
        axes[1].bar(dates, -df['discharge_kw'],
                    width=pd.Timedelta('15min'), color='red',   alpha=0.7, label='Discharge')
        axes[1].axhline( max_charge_kw,    color='green', linestyle='--', linewidth=1.0,
                         label=f'Max charge ({max_charge_kw} kW)')
        axes[1].axhline(-max_discharge_kw, color='red',   linestyle='--', linewidth=1.0,
                         label=f'Max discharge ({max_discharge_kw} kW)')
        axes[1].set_ylabel('Power (kW)')
        axes[1].set_xlabel('Date')
        axes[1].legend(loc='upper right', fontsize=8)
        axes[1].set_title('Charge / Discharge')
        axes[1].grid(True, alpha=0.3)

        axes[1].xaxis.set_major_locator(mdates.MonthLocator())
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        fig.autofmt_xdate(rotation=45)
        plt.tight_layout()
        plt.show()


## 8. REC + Battery Summary

The monthly analysis table reports the key operational metrics that quantify the joint effect of REC membership and battery storage optimisation:

- **Internal shared energy (MWh):** Total electricity traded within the community at the reduced grid access tariff, disaggregated by month.  Higher values during summer reflect increased PV surplus available for internal sharing; battery storage extends this surplus into evening hours.
- **Battery throughput (MWh):** Total charge and discharge energy per battery asset, including round-trip efficiency losses.  The ratio of discharge to charge (< 1.0 due to $\eta^{\mathrm{ch}} \cdot \eta^{\mathrm{dis}} = 0.90$) quantifies energy lost to storage cycling.

These metrics serve as inputs to the cross-scenario comparison (C3 vs A2-mixed, C3 vs A1-mixed, C3 vs B2-mixed) that isolates the marginal contribution of battery dispatch to community self-sufficiency and cost reduction.

In [ ]:
print("=== C3 REC + Battery Summary ===")
print(f"Total Shared Energy: {pipe.es_monthly_analysis_df['internal_shared_energy_mwh'].sum():,.2f} MWh")

schedules = getattr(pipe, 'battery_schedules', None) or {}
if not schedules and hasattr(pipe, 'battery_schedule_df') and not pipe.battery_schedule_df.empty:
    schedules = {'BESS_REC_01': pipe.battery_schedule_df}

for batt_id, df in schedules.items():
    total_ch  = df['charge_kw'].sum()    * 0.25 / 1000
    total_dis = df['discharge_kw'].sum() * 0.25 / 1000
    print(f"  [{batt_id}] Charge: {total_ch:,.2f} MWh  |  Discharge: {total_dis:,.2f} MWh")

pipe.es_monthly_analysis_df


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Normalise: build {battery_id: (schedule_df, tech_params)} from all available batteries
batteries_raw = pipe.config.get('batteries', pipe.config.get('battery_storage'))
if isinstance(batteries_raw, dict):
    batteries_raw = [batteries_raw]

battery_cfgs = {b['battery_id']: b.get('technical_parameters', {}) for b in batteries_raw}

# Use per-battery schedules if available, fall back to single aggregated schedule
schedules = getattr(pipe, 'battery_schedules', None) or {}
if not schedules and hasattr(pipe, 'battery_schedule_df') and not pipe.battery_schedule_df.empty:
    batt_id = list(battery_cfgs.keys())[0]
    schedules = {batt_id: pipe.battery_schedule_df}

if not schedules:
    print("Battery optimization results not available")
else:
    for batt_id, df in schedules.items():
        tech = battery_cfgs.get(batt_id, {})
        capacity_kwh     = tech.get('capacity_kwh', 200)
        max_charge_kw    = tech.get('max_charge_power_kw', 68)
        max_discharge_kw = tech.get('max_discharge_power_kw', 68)
        soc_min_kwh      = capacity_kwh * tech.get('soc_min_percent', 20)  / 100.0
        soc_max_kwh      = capacity_kwh * tech.get('soc_max_percent', 100) / 100.0
        initial_soc_kwh  = capacity_kwh * tech.get('initial_soc_percent', 50) / 100.0

        batt_name = next((b['battery_name'] for b in batteries_raw if b['battery_id'] == batt_id), batt_id)

        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        fig.suptitle(f'{batt_id} — {batt_name}', fontsize=12, fontweight='bold')

        dates = df.index

        axes[0].plot(dates, df['soc_kwh'], label='SOC', color='blue', linewidth=0.8)
        axes[0].axhline(soc_max_kwh,     color='red',    linestyle='--', linewidth=1.0,
                        label=f'SOC max ({soc_max_kwh:.0f} kWh)')
        axes[0].axhline(soc_min_kwh,     color='orange', linestyle='--', linewidth=1.0,
                        label=f'SOC min ({soc_min_kwh:.0f} kWh)')
        axes[0].axhline(initial_soc_kwh, color='gray',   linestyle=':',  linewidth=1.0,
                        label=f'Initial SOC ({initial_soc_kwh:.0f} kWh)')
        axes[0].set_ylabel('SOC (kWh)')
        axes[0].legend(loc='upper right', fontsize=8)
        axes[0].set_title(f'State of Charge  [capacity: {capacity_kwh} kWh]')
        axes[0].grid(True, alpha=0.3)

        axes[1].bar(dates, df['charge_kw'],
                    width=pd.Timedelta('15min'), color='green', alpha=0.7, label='Charge')
        axes[1].bar(dates, -df['discharge_kw'],
                    width=pd.Timedelta('15min'), color='red',   alpha=0.7, label='Discharge')
        axes[1].axhline( max_charge_kw,    color='green', linestyle='--', linewidth=1.0,
                         label=f'Max charge ({max_charge_kw} kW)')
        axes[1].axhline(-max_discharge_kw, color='red',   linestyle='--', linewidth=1.0,
                         label=f'Max discharge ({max_discharge_kw} kW)')
        axes[1].set_ylabel('Power (kW)')
        axes[1].set_xlabel('Date')
        axes[1].legend(loc='upper right', fontsize=8)
        axes[1].set_title('Charge / Discharge')
        axes[1].grid(True, alpha=0.3)

        axes[1].xaxis.set_major_locator(mdates.MonthLocator())
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        fig.autofmt_xdate(rotation=45)
        plt.tight_layout()
        plt.show()


In [ ]:
# REC + Battery Analysis
print("=== C3 Summary ===")
print(f"Total Shared Energy: {pipe.es_monthly_analysis_df['internal_shared_energy_mwh'].sum():,.2f} MWh")

schedules = getattr(pipe, 'battery_schedules', None) or {}
if not schedules and hasattr(pipe, 'battery_schedule_df') and not pipe.battery_schedule_df.empty:
    schedules = {'BESS_REC_01': pipe.battery_schedule_df}

for batt_id, df in schedules.items():
    total_ch  = df['charge_kw'].sum()    * 0.25 / 1000
    total_dis = df['discharge_kw'].sum() * 0.25 / 1000
    print(f"  [{batt_id}] Charge: {total_ch:,.2f} MWh  |  Discharge: {total_dis:,.2f} MWh")

pipe.es_monthly_analysis_df
